In [0]:
# =====================================================
# Imports
# =====================================================

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    when,
    lit,
    to_date,
    avg,
    min,
    max,
    sum,
    count,
    round,
    current_timestamp,
    trunc,
    year,
    month,
    date_format
)

In [0]:
# =====================================================
# Constants
# =====================================================

SILVER_TABLE_NAME = "weather_project.north_texas_weather.silver_hourly_weather"

GOLD_DAILY_TABLE_NAME = "weather_project.north_texas_weather.gold_daily_weather_summary"

GOLD_EXTREME_TABLE_NAME = "weather_project.north_texas_weather.gold_extreme_weather_events"

GOLD_MONTHLY_TABLE_NAME = "weather_project.north_texas_weather.gold_monthly_trends"

EXTREME_HEAT_THRESHOLD_F = 100.0

FREEZE_THRESHOLD_F = 32.0

HEAVY_PRECIPITATION_THRESHOLD_IN = 2.0

In [0]:
# =====================================================
# Configure Spark
# =====================================================

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

In [0]:
# =====================================================
# Runtime parameters
# =====================================================

dbutils.widgets.text(
    "ingestion_id",
    ""
)

dbutils.widgets.text(
    "has_new_data",
    "true"
)

INGESTION_ID = (
    dbutils.widgets
    .get("ingestion_id")
    .strip()
)

HAS_NEW_DATA = (
    dbutils.widgets
    .get("has_new_data")
    .strip()
    .lower()
)

# =====================================================
# Handle successful no-data Bronze runs
# =====================================================

if HAS_NEW_DATA == "false":
    dbutils.notebook.exit("NO_NEW_DATA_FOR_GOLD")

if not INGESTION_ID:
    raise RuntimeError("Gold requires an ingestion_id from the upstream pipeline")

In [0]:
# =====================================================================
# Obtain affected location-days from silver table
# =====================================================================

# =====================================================
# 1. Read trusted Silver data
# =====================================================

silver_df = (
    spark.read
    .table(SILVER_TABLE_NAME)
    .withColumn(
        "observation_date_utc",
        to_date(col("timestamp_utc"))
    )
)

# =====================================================
# 2. Determine affected location-days
# =====================================================

if INGESTION_ID.upper() == "ALL":

    # Initial Gold Build
    affected_location_dates_df = (
        silver_df
        .select(
            "location_id",
            "observation_date_utc"
        )
        .distinct()
    )

else:

    # Production path: identify only location-days touched by latest silver ingestion
    current_ingestion_df = (
        silver_df
        .filter(
            col("ingestion_id") == INGESTION_ID
        )
    )

    if current_ingestion_df.limit(1).count() == 0:
        dbutils.notebook.exit("NO_SILVER_ROWS_FOR_INGESTION")

    affected_location_dates_df = (
        current_ingestion_df
        .select(
            "location_id",
            "observation_date_utc"
        )
        .distinct()
    )

# =====================================================================
# 3. Retrieve complete Silver history for affected location-days
# =====================================================================

affected_silver_df = (
    silver_df
    .join(
        affected_location_dates_df,
        on=[
            "location_id",
            "observation_dates_utc"
        ],
        how="left_semi"
    )
)

In [0]:
# =====================================================
# Daily weather summary for all cities
# Includes max, min, and total precipitation for each city
# Grain: One row per location_id per UTC calendar date
# =====================================================

daily_weather_summary_df = (
    affected_silver_df
    .groupBy(
        "location_id",
        "city",
        "state_code",
        "country_code",
        "observation_date_utc"
    )
    .agg(
        round(avg("temperature_celsius"), 2).alias("avg_temperature_celsius"),
        round(min("temperature_celsius"), 2).alias("min_temperature_celsius"),
        round(max("temperature_celsius"), 2).alias("max_temperature_celsius"),
        round(avg("temperature_fahrenheit"), 2).alias("avg_temperature_fahrenheit"),
        round(min("temperature_fahrenheit"), 2).alias("min_temperature_fahrenheit"),
        round(max("temperature_fahrenheit"), 2).alias("max_temperature_fahrenheit"),
        round(sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        round(sum("precipitation_inches"), 2).alias("total_precipitation_inches"),
        count("*").alias("hourly_observation_count"),
        max("ingested_at").alias("latest_source_ingested_at")
    )
    .withColumn(
        "gold_updated_at",
        current_timestamp()
    )
)

# =====================================================
# Create Daily Weather Summary Table
# =====================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS
weather_project.north_texas_weather.gold_daily_weather_summary
(
    location_id STRING NOT NULL,

    city STRING NOT NULL,
    state_code STRING NOT NULL,
    country_code STRING NOT NULL,

    observation_date_utc DATE NOT NULL,

    avg_temperature_celsius DOUBLE,
    min_temperature_celsius DOUBLE,
    max_temperature_celsius DOUBLE,

    avg_temperature_fahrenheit DOUBLE,
    min_temperature_fahrenheit DOUBLE,
    max_temperature_fahrenheit DOUBLE,

    total_precipitation_mm DOUBLE,
    total_precipitation_inches DOUBLE,

    hourly_observation_count BIGINT,

    latest_source_ingested_at TIMESTAMP,
    gold_updated_at TIMESTAMP NOT NULL
)
USING DELTA
""")

gold_daily_target = DeltaTable.forName(
    spark,
    GOLD_DAILY_TABLE_NAME
)

(
    gold_daily_target
    .alias("target")

    .merge(
        daily_weather_summary_df.alias("source"),

        """
        target.location_id =
            source.location_id

        AND target.observation_date_utc =
            source.observation_date_utc
        """
    )

    .whenMatchedUpdateAll()

    .whenNotMatchedInsertAll()

    .execute()
)

display(
    daily_weather_summary_df
    .orderBy(
        "city",
        "observation_date_utc"
    )
)

print(
    "Gold daily weather summary "
    "successfully updated."
)

In [0]:
# =====================================================
# Extreme weather events
# =====================================================

# =====================================================
# Classify affected location-days
# =====================================================

extreme_weather_candidate_df = (
    affected_daily_weather_df
    .withColumn(
        "is_extreme_heat",
        col("max_temperature_fahrenheit") >= EXTREME_HEAT_THRESHOLD_F
    )
    .withColumn(
        "is_freeze_event",
        col("min_temperature_fahrenheit") <= FREEZE_THRESHOLD_F
    )
    .withColumn(
        "is_heavy_precipitation",
        col("total_precipitation_inches") >= HEAVY_PRECIPITATION_THRESHOLD_IN
    )
    .withColumn(
        "is_extreme_event",
        col("is_extreme_heat") | col("is_freeze_event") | col("is_heavy_precipitation")
    )
    .withColumn(
        "extreme_updated_at",
        current_timestamp()
    )
)

extreme_weather_merge_df = (
    extreme_weather_candidate_df
    .select(
        col("location_id"),
        col("city"),
        col("state_code"),
        col("country_code"),
        col("observation_date_utc"),
        col("avg_temperature_fahrenheit"),
        col("min_temperature_fahrenheit"),
        col("max_temperature_fahrenheit"),
        col("total_precipitation_inches"),
        col("avg_temperature_celsius"),
        col("min_temperature_celsius"),
        col("max_temperature_celsius"),
        col("total_precipitation_mm"),
        col("hourly_observation_count"),
        col("is_extreme_heat"),
        col("is_freeze_event"),
        col("is_heavy_precipitation"),
        col("latest_source_ingested_at"),
        col("extreme_updated_at")
    )
)

# =====================================================
# Create Extreme Weather Events Table
# =====================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS
weather_project.north_texas_weather.gold_extreme_weather_events
(
    location_id STRING NOT NULL,

    city STRING NOT NULL,
    state_code STRING NOT NULL,
    country_code STRING NOT NULL,

    observation_date_utc DATE NOT NULL,

    avg_temperature_fahrenheit DOUBLE,
    min_temperature_fahrenheit DOUBLE,
    max_temperature_fahrenheit DOUBLE,

    total_precipitation_inches DOUBLE,

    hourly_observation_count BIGINT,

    is_extreme_heat BOOLEAN NOT NULL,
    is_freeze_event BOOLEAN NOT NULL,
    is_heavy_precipitation BOOLEAN NOT NULL,
    is_extreme_event BOOLEAN NOT NULL,

    latest_source_ingested_at TIMESTAMP,
    extreme_updated_at TIMESTAMP NOT NULL
)
USING DELTA
""")

extreme_weather_target = DeltaTable.forName(spark, GOLD_EXTREME_TABLE_NAME)

(
    extreme_weather_target
    .alias("target")

    .merge(
        extreme_weather_merge_df.alias("source"),

        """
        target.location_id =
            source.location_id

        AND target.observation_date_utc =
            source.observation_date_utc
        """
    )

    # A previously extreme day was recalculated
    # and no longer satisfies any extreme rule.
    .whenMatchedDelete(
        condition="""
        source.is_extreme_event = false
        """
    )

    # Existing event remains extreme:
    # refresh it with the latest daily metrics.
    .whenMatchedUpdateAll(
        condition="""
        source.is_extreme_event = true
        """
    )

    # Newly identified extreme event.
    .whenNotMatchedInsertAll(
        condition="""
        source.is_extreme_event = true
        """
    )

    .execute()
)

display(
    extreme_weather_merge_df
    .orderBy(
        col("observation_date_utc").desc(),
        col("city")
    )
)

print(
    "Gold Extreme Weather Events "
    "successfully updated."
)

In [0]:
# =====================================================
# Monthly Trends
# =====================================================

# =====================================================
# Determine affected location-months
#
# daily_weather_summary_df contains only the location-days
# recalculated during the current pipeline run.
# =====================================================

affected_location_months_df = (
    daily_weather_summary_df
    .select(
        col("location_id"),
        trunc(
            col("observation_date_utc"),
            "month"
        ).alias("month_start") # Map to month start
    )
    .distinct()
)

# =====================================================
# Read complete daily Gold data
# for the affected location-months
# =====================================================

# Must read again from month start to latest day of the month from gold data

daily_gold_df = (
    spark.read
    .table(GOLD_DAILY_TABLE_NAME)
    .withColumn(
        "month_start",
        trunc(
            col("observation_date_utc"),
            "month"
        )
    )
)

affected_month_daily_df = (
    daily_gold_df
    .join(
        affected_location_months_df,
        on=[
            "location_id",
            "month_start"
        ],
        how="left_semi"
    )
)

# =====================================================
# Build Gold monthly weather trends
#
# Grain:
# One row per location_id per UTC calendar month
# =====================================================

monthly_trends_df = (
    affected_month_daily_df

    .groupBy(
        "location_id",
        "city",
        "state_code",
        "country_code",
        "month_start"
    )

    .agg(
        round(
            avg("max_temperature_celsius"),
            2
        ).alias(
            "avg_high_temperature_celsius"
        ),

        round(
            avg("min_temperature_celsius"),
            2
        ).alias(
            "avg_low_temperature_celsius"
        ),

        round(
            avg("max_temperature_fahrenheit"),
            2
        ).alias(
            "avg_high_temperature_fahrenheit"
        ),

        round(
            avg("min_temperature_fahrenheit"),
            2
        ).alias(
            "avg_low_temperature_fahrenheit"
        ),

        round(
            sum("total_precipitation_mm"),
            2
        ).alias(
            "total_monthly_precipitation_mm"
        ),

        round(
            sum("total_precipitation_inches"),
            2
        ).alias(
            "total_monthly_precipitation_inches"
        ),

        count("*").alias(
            "days_observed"
        ),

        sum(
            "hourly_observation_count"
        ).alias(
            "hourly_observation_count"
        ),

        max(
            "latest_source_ingested_at"
        ).alias(
            "latest_source_ingested_at"
        )
    )

    .withColumn(
        "year",
        year(col("month_start"))
    )

    .withColumn(
        "month_number",
        month(col("month_start"))
    )

    .withColumn(
        "month_name",
        date_format(
            col("month_start"),
            "MMMM"
        )
    )

    .withColumn(
        "gold_updated_at",
        current_timestamp()
    )
)

# =====================================================
# Define persistent Gold monthly table contract
# =====================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS
weather_project.north_texas_weather.gold_monthly_trends
(
    location_id STRING NOT NULL,

    city STRING NOT NULL,
    state_code STRING NOT NULL,
    country_code STRING NOT NULL,

    month_start DATE NOT NULL,

    year INT NOT NULL,
    month_number INT NOT NULL,
    month_name STRING NOT NULL,

    avg_high_temperature_celsius DOUBLE,
    avg_low_temperature_celsius DOUBLE,

    avg_high_temperature_fahrenheit DOUBLE,
    avg_low_temperature_fahrenheit DOUBLE,

    total_monthly_precipitation_mm DOUBLE,
    total_monthly_precipitation_inches DOUBLE,

    days_observed BIGINT,
    hourly_observation_count BIGINT,

    latest_source_ingested_at TIMESTAMP,
    gold_updated_at TIMESTAMP NOT NULL
)
USING DELTA
""")

monthly_target = DeltaTable.forName(
    spark,
    GOLD_MONTHLY_TABLE_NAME
)

(
    monthly_target
    .alias("target")

    .merge(
        monthly_trends_df.alias("source"),

        """
        target.location_id =
            source.location_id

        AND target.month_start =
            source.month_start
        """
    )

    .whenMatchedUpdateAll()

    .whenNotMatchedInsertAll()

    .execute()
)

display(
    monthly_trends_df
    .orderBy(
        "city",
        "month_start"
    )
)

print(
    "Gold Monthly Trends "
    "successfully updated."
)

In [0]:
# Monthly trends

from pyspark.sql.functions import to_date, col, max, min, sum, round, month, avg, year, date_format

df_daily_weather_summary = spark.read.table("weather_project.north_texas_weather.gold_daily_weather_summary")

df_monthly_trends = df_daily_weather_summary \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", date_format(col("date"), "MMMM"))

df_monthly_trends = df_monthly_trends.groupBy("city", "year", "month").agg(
    round(avg("max_temperature_celsius"), 2).alias("avg_high_temperature_celsius"),
    round(avg("min_temperature_celsius"), 2).alias("avg_low_temperature_celsius"),
    round(avg("max_temperature_fahrenheit"), 2).alias("avg_high_temperature_fahrenheit"),
    round(avg("min_temperature_fahrenheit"), 2).alias("avg_low_temperature_fahrenheit"),
    round(sum("total_precipitation_inches"), 2).alias("total_monthly_precipitation_inches")
)

df_monthly_trends = df_monthly_trends.orderBy("city", "year")

display(df_monthly_trends)

df_monthly_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("weather_project.north_texas_weather.gold_monthly_trends")

print("Successfully built and saved the Gold Monthly Trends!")